# Updating UMAPs

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import scanpy as sc
import infercnvpy as cnv
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import muon as mu


sc.settings.verbosity = 3
plt.rcParams['figure.dpi'] = 200
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['figure.figsize'] = (5, 5)
plt.rcParams['pdf.fonttype'] = 'truetype'

## Updating patient colors

In [ ]:
adata = sc.read_h5ad("MDS_adata_for_publication_v2.h5ad")

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(5,5))

sc.pl.umap(adata, color="patient_alias", ax=ax, show=False)
ax.set_rasterization_zorder(0)
fig.savefig("figures/patient_alias_new_colors_UMAP.pdf", dpi=200, bbox_inches = "tight")
plt.close()

## Updating progression plots per patient

In [ ]:
adata.obs['disease_state'].value_counts()

In [ ]:
newdf=pd.DataFrame({
    "X_coord_umap":adata.obsm["X_umap"][:,0],
    "Y_coord_umap":adata.obsm["X_umap"][:,1],
    "celltype":adata.obs["celltype_v2"],
    "outcome_C6D28":adata.obs["outcome_C6D28"],
    'outcome_C12D29':adata.obs["outcome_C12D29"],
    "timepoint_coarse":adata.obs["timepoint_coarse"],
    "patient":adata.obs["patient"],
    "patient_alias": adata.obs["patient_alias"],
})
newdf

In [ ]:
from MDS_figure2_dicts import *

In [ ]:
patient_alias_colors

In [ ]:
newdf['ctgrey'] = "#c8c8c8"

In [ ]:
titles1=['C1D1','C7D1','C12D29']
titles2=['C1D1','C7D1','AML Progression post C6']

### Responders

In [ ]:
import matplotlib.lines as mlines

# Row 1

fig, axes = plt.subplots(1, 3, figsize=(12, 4), dpi=300)

# List to store unique patient aliases and their associated colors for the legend
unique_patient_aliases = []

for j, timepoint in enumerate(['pre', 'mid', 'post']):
    ax = axes[j]
    
    # Background scatter plot (grey points)
    ax.scatter(
        newdf['X_coord_umap'], 
        newdf['Y_coord_umap'], 
        c=newdf['ctgrey'], 
        s=0.5, 
        rasterized=True  # Rasterizing background points
    )
    
    # Filtering data
    df = newdf[newdf['outcome_C6D28'] == "Responder"]
    data = df[(df['outcome_C12D29'] == "Responder") & (df['timepoint_coarse'] == timepoint)]
    data['patient_alias'] = data['patient_alias'].cat.remove_unused_categories()

    # Plotting the filtered data
    scatter = ax.scatter(
        data['X_coord_umap'], 
        data['Y_coord_umap'], 
        c=[patient_alias_colors[alias] for alias in data['patient_alias']], 
        s=3,
        edgecolor='none',
        rasterized=True  # Enable rasterization for the scatter plot
    )

    # Collect unique patient aliases across all timepoints
    unique_patient_aliases.extend(data['patient_alias'].unique())

    # Set title
    plottitle = 'Responder - ' + titles1[j]
    ax.set_title(plottitle)
    
    # Remove grid and axis labels
    ax.grid(False)
    ax.legend().set_visible(False)  # Hide subplot legends
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel('')
    ax.set_ylabel('')

# Remove duplicates from the unique_patient_aliases list
unique_patient_aliases = list(set(unique_patient_aliases))

# Create the legend handles for unique patients
legend_handles = [
    mlines.Line2D([], [], marker='o', color='w', markerfacecolor=patient_alias_colors[alias], markersize=3, label=alias)
    for alias in unique_patient_aliases
]

# Create a single legend outside the figure using the stored handles
fig.legend(
    legend_handles, 
    [handle.get_label() for handle in legend_handles],  # Extracting the labels from the handles
    title='Patient', 
    markerscale=7, 
    fontsize=20, 
    loc='upper center', 
    bbox_to_anchor=(0.5, -0.1), 
    ncol=2
)

# Adjust layout and save with rasterization enabled
fig.tight_layout()
fig.savefig("figures/row1_responder_responder_UMAP_v2.png", format="png", bbox_inches='tight')
fig.savefig("figures/row1_responder_responder_UMAP_v2.pdf", format="pdf", bbox_inches='tight')


In [ ]:
# Row 2
fig, axes = plt.subplots(1, 3, figsize=(12, 4), dpi=300)

# List to store unique patient aliases and their associated colors for the legend
unique_patient_aliases = []

for j, timepoint in enumerate(['pre', 'mid', 'progression']):
    ax = axes[j]
    
    # Background scatter plot (grey points)
    ax.scatter(
        newdf['X_coord_umap'], 
        newdf['Y_coord_umap'], 
        c=newdf['ctgrey'], 
        s=0.5, 
        rasterized=True  # Rasterizing background points
    )
    
    # Filtering data
    df = newdf[newdf['outcome_C6D28'] == "Responder"]
    data = df[(df['outcome_C12D29'] == "Progression") & (df['timepoint_coarse'] == timepoint)]
    data['patient_alias'] = data['patient_alias'].cat.remove_unused_categories()
    
    # Plotting the filtered data
    scatter = ax.scatter(
        data['X_coord_umap'], 
        data['Y_coord_umap'], 
        c=[patient_alias_colors[alias] for alias in data['patient_alias']], 
        s=3,
        edgecolor='none',
        rasterized=True  # Enable rasterization for the scatter plot
    )

    # Collect unique patient aliases across all timepoints
    unique_patient_aliases.extend(data['patient_alias'].unique())

    # Set title
    plottitle = 'Responder - ' + titles2[j]
    ax.set_title(plottitle)
    
    # Remove grid and axis labels
    ax.grid(False)
    ax.legend().set_visible(False)  # Hide subplot legends
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel('')
    ax.set_ylabel('')

# Remove duplicates from the unique_patient_aliases list
unique_patient_aliases = list(set(unique_patient_aliases))

# Create the legend handles for unique patients
legend_handles = [
    mlines.Line2D([], [], marker='o', color='w', markerfacecolor=patient_alias_colors[alias], markersize=3, label=alias)
    for alias in unique_patient_aliases
]

# Create a single legend outside the figure using the stored handles
fig.legend(
    legend_handles, 
    [handle.get_label() for handle in legend_handles],  # Extracting the labels from the handles
    title='Patient', 
    markerscale=7, 
    fontsize=20, 
    loc='upper center', 
    bbox_to_anchor=(0.5, -0.1), 
    ncol=3
)

# Adjust layout and save with rasterization enabled
fig.tight_layout()
fig.savefig("figures/row2_responder_progression_UMAP_v2.png", format="png", bbox_inches='tight', dpi=300)
fig.savefig("figures/row2_responder_progression_UMAP_v2.pdf", format="pdf", bbox_inches='tight')


### Non-responders

In [ ]:
#Row 3
fig, axes = plt.subplots(1, 3, figsize=(12, 4), dpi=300)
ptt = ['Non-Responder - C1D1', 'Non-Responder - C7D1', 'Responder - C12D29']

# List to store unique patient aliases and their associated colors for the legend
unique_patient_aliases = []

for j, timepoint in enumerate(['pre', 'mid', 'post']):
    ax = axes[j]
    
    # Background scatter plot (grey points)
    ax.scatter(
        newdf['X_coord_umap'], 
        newdf['Y_coord_umap'], 
        c=newdf['ctgrey'], 
        s=0.5, 
        rasterized=True  # Rasterizing background points
    )
    
    # Filtering data
    df = newdf[newdf['outcome_C6D28'] == "Non-Responder"]
    data = df[(df['outcome_C12D29'] == "Responder") & (df['timepoint_coarse'] == timepoint)]
    data['patient_alias'] = data['patient_alias'].cat.remove_unused_categories()
    
    # Plotting the filtered data
    scatter = ax.scatter(
        data['X_coord_umap'], 
        data['Y_coord_umap'], 
        c=[patient_alias_colors[alias] for alias in data['patient_alias']], 
        s=3,
        edgecolor='none',
        rasterized=True  # Enable rasterization for the scatter plot
    )

    # Collect unique patient aliases across all timepoints
    unique_patient_aliases.extend(data['patient_alias'].unique())

    # Set title
    ax.set_title(ptt[j])
    
    # Remove grid and axis labels
    ax.grid(False)
    ax.legend().set_visible(False)  # Hide subplot legends
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel('')
    ax.set_ylabel('')

# Remove duplicates from the unique_patient_aliases list
unique_patient_aliases = list(set(unique_patient_aliases))

# Create the legend handles for unique patients
legend_handles = [
    mlines.Line2D([], [], marker='o', color='w', markerfacecolor=patient_alias_colors[alias], markersize=3, label=alias)
    for alias in unique_patient_aliases
]

# Create a single legend outside the figure using the stored handles
fig.legend(
    legend_handles, 
    [handle.get_label() for handle in legend_handles],  # Extracting the labels from the handles
    title='Patient', 
    markerscale=7, 
    fontsize=20, 
    loc='upper center', 
    bbox_to_anchor=(0.5, -0.1), 
    ncol=3
)

# Adjust layout and save with rasterization enabled
fig.tight_layout()
fig.savefig("figures/row3_nonresponder_responder_UMAP_v2.png", format="png", bbox_inches='tight')
fig.savefig("figures/row3_nonresponder_responder_UMAP_v2.pdf", format="pdf", bbox_inches='tight')


In [ ]:
#Row 4

fig, axes = plt.subplots(1, 3, figsize=(12, 4), dpi=300)

# List to store unique patient aliases and their associated colors for the legend
unique_patient_aliases = []

for j, timepoint in enumerate(['pre', 'mid', 'progression']):
    ax = axes[j]
    
    # Background scatter plot (grey points)
    ax.scatter(
        newdf['X_coord_umap'], 
        newdf['Y_coord_umap'], 
        c=newdf['ctgrey'], 
        s=0.5, 
        rasterized=True  # Rasterized background points
    )
    
    # Filter relevant data
    df = newdf[newdf['outcome_C6D28'] == "Non-Responder"]
    data = df[(df['outcome_C12D29'] == "Progression") & (df['timepoint_coarse'] == timepoint)]
    data['patient_alias'] = data['patient_alias'].cat.remove_unused_categories()
    
    # Foreground scatter plot with hue
    scatter = sns.scatterplot(
        data=data, 
        x='X_coord_umap', 
        y='Y_coord_umap', 
        ax=ax, 
        hue='patient_alias', 
        palette=patient_alias_colors, 
        s=3,
        edgecolor='none'
    )

    # Collect unique patient aliases across all timepoints
    unique_patient_aliases.extend(data['patient_alias'].unique())

    # Apply rasterization to foreground scatter points
    for collection in scatter.collections:
        collection.set_rasterized(True)  

    # Set title
    ax.set_title(f'Non-responders - {titles2[j]}')
    
    # Remove grid and axis labels
    ax.grid(False)
    ax.legend().set_visible(False)  # Hide subplot legends
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel('')
    ax.set_ylabel('')

# Remove duplicates from the unique_patient_aliases list
unique_patient_aliases = list(set(unique_patient_aliases))

# Create the legend handles for unique patients
legend_handles = [
    mlines.Line2D([], [], marker='o', color='w', markerfacecolor=patient_alias_colors[alias], markersize=3, label=alias)
    for alias in unique_patient_aliases
]

# Create a single legend outside the figure using the stored handles
fig.legend(
    legend_handles, 
    [handle.get_label() for handle in legend_handles],  # Extracting the labels from the handles
    title='Patient', 
    markerscale=7, 
    fontsize=20, 
    loc='upper center', 
    bbox_to_anchor=(0.5, -0.1), 
    ncol=3
)

# Adjust layout and save with rasterization enabled
fig.tight_layout()
fig.savefig("figures/row4_nonresponders_progression_UMAP_v2.png", format="png", bbox_inches='tight')
fig.savefig("figures/row4_nonresponders_progression_UMAP_v2.pdf", format="pdf", bbox_inches='tight')


## Healthy celltypes in grey to highlight ATC

In [ ]:
healthy_ct = ["HSC", 'MPP', 'MEP/MKP', 'MEP', 'Erythroblast', 'BMCP','GMP','Monocyte progenitor']

In [ ]:
adata.obs['celltype_v2'].cat.categories

In [ ]:
 celltype_v2_with_grey= ['#023fa5', '#7d87b9', '#bec1d4', '#d6bcc0', '#bb7784', '#8e063b',
        '#4a6fe3', '#8595e1', '#b5bbe3', '#e6afb9', '#e07b91', '#d33f6a',
        '#11c638', '#8dd593', '#c6dec7', '#808080', '#808080', '#808080',
        '#808080', '#808080', '#808080', '#808080', '#808080']

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(5,5))

sc.pl.umap(adata, color="celltype_v2", palette=celltype_v2_with_grey, legend_loc="on data", show=False, ax=ax, legend_fontsize=6)
ax.set_rasterization_zorder(0)
fig.savefig("figures/celltype_v2_healthyingrey_UMAP.pdf", dpi=200, bbox_inches = "tight")
#plt.close()



## Healthy patients in grey to highlight MDS patients

In [ ]:
adata.obs['patient_alias'] = adata.obs['patient_alias'].cat.reorder_categories(['P08', 'P12', 'P03', 'P09', 'P02', 'P01', 'P11', 'P17', 'P18','C1','C2','C3','C4','C5'])

In [ ]:
patient_alias_colors_withgrey = {
    "P01":"#1f77b4",
    "P02":"#ff7f0e",
    "P03":"#d62728",
    "P08":"#9467bd",
    "P09":"#2ca02c",
    "P11":"#8c564b",
    "P12":"#bcbd22",
    "P17":"#e377c2",
    "P18":"#17becf",
    'C1': '#808080',
    'C2': '#808080',
    'C3': '#808080',
    'C4': '#808080',
    'C5': '#808080'
}

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(5,5))
sc.pl.umap(adata, color='patient_alias', palette=patient_alias_colors_withgrey, ax=ax, show=False)
ax.set_rasterization_zorder(0)
fig.savefig("figures/patient_alias_healthyingrey_UMAP.pdf", dpi=200, bbox_inches = "tight")

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(5,5))

sc.pl.umap(adata, color="celltype_v2", palette=celltype_v2_with_grey, legend_loc="on data", show=False, ax=ax, legend_fontsize=6)
ax.set_rasterization_zorder(0)
fig.savefig("figures/celltype_v2_healthyingrey_UMAP.pdf", dpi=200, bbox_inches = "tight")
#plt.close()



## Atypical clusters in grey

In [ ]:
adata.obs["celltype_v2"].cat.categories

In [ ]:
celltype_v2_colors_original = adata.uns['celltype_v2_colors']

In [ ]:
celltype_v2_atypicalgrey = ['#808080',
 '#808080',
 '#808080',
 '#808080',
 '#808080',
 '#808080',
 '#808080',
 '#808080',
 '#808080',
 '#808080',
 '#808080',
 '#808080',
 '#808080',
 '#808080',
 '#808080',
 '#ead3c6',
 '#f0b98d',
 '#ef9708',
 '#0fcfc0',
 '#9cded6',
 '#d5eae7',
 '#f3e1eb',
 '#f6c4e1']

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(5,5))

sc.pl.umap(adata, color="celltype_v2", palette=celltype_v2_atypicalgrey, legend_loc="on data", show=False, ax=ax, legend_fontsize=6)
ax.set_rasterization_zorder(0)
fig.savefig("figures/celltype_v2_atypicalingrey_UMAP.pdf", dpi=200, bbox_inches = "tight")
#plt.close()



# MISC

In [ ]:
temp_df = pd.DataFrame({
    "patient_alias":adata.obs["patient_alias"],
    "timepoint_coarse":adata.obs["timepoint_coarse"],
    "celltype_v2":adata.obs["celltype_v2"]
})
temp_df

In [ ]:
grouped_counts = temp_df.groupby(['patient_alias', 'timepoint_coarse', 'celltype_v2']).size().reset_index(name='count')
grouped_counts = grouped_counts[grouped_counts['count']>0].copy()
grouped_counts

In [ ]:
grouped_counts.to_csv("MDS_paper_grouped_counts.csv", index=False)

# New celltype colours

In [ ]:
new_celltype_colors = {
    "Atypical cluster a": "#e2b9c4",
    "Atypical cluster c": "#9e163c",
    "Atypical cluster b": "#c5738a",
    "Atypical cluster d": "#905898",
    "Atypical cluster f": "#bf565d",
    "Atypical cluster e": "#cb7b26",
    "Atypical cluster g": "#a6c1d5",
    "Atypical cluster h": "#4d83ab",
    "Atypical cluster i": "#cbc106",
    "Atypical cluster k": "#7b906f",
    "Atypical cluster l": "#bbd2c6",
    "Atypical cluster n": "#1c6840",
    "Atypical cluster m": "#77a48c",
    "Atypical cluster j": "#389ca7",
    "Atypical cluster o": "#ffffff",
    "HSC": "#c098b0",
    "MPP": "#583070",
    "MEP/MKP": "#ffb651",
    "MEP": "#d85a44",
    "Erythroblast": "#aa3f5d",
    "BMCP": "#bcd1d4",
    "GMP": "#79a3aa",
    "Monocyte progenitor": "#206671"
}

other_celltype_colors={
    "Patient specific cluster": "#808080",
    "Healthy cluster": "#d3d3d3",
    "Atypical cluster": "#808080"
}

misc_colors={
   "Responder": "#115284",
    "Non-responder": "#fe9003",
    "Baseline": "#515558"
}

all_colors= new_celltype_colors | other_celltype_colors | misc_colors

In [ ]:
atypical_ct=[
    "Atypical cluster a",
    "Atypical cluster b",
    "Atypical cluster c",
    "Atypical cluster d",
    "Atypical cluster e",
    "Atypical cluster f",
    "Atypical cluster g",
    "Atypical cluster h",
    "Atypical cluster i",
    "Atypical cluster j",
    "Atypical cluster k",
    "Atypical cluster l",
    "Atypical cluster m",
    "Atypical cluster n",
    "Atypical cluster o",
]

In [ ]:
fig, ax = plt.subplots(figsize=(6,6))
sc.pl.umap(adata, 
           color=['celltype_v2'], 
           groups=healthy_ct,
           palette=all_colors, 
           na_color="#808080", 
           ax=ax, 
           show=False, 
           title="healthy colored",
           frameon=False)
ax.set_rasterization_zorder(0)
fig.savefig("figures/mastercolors_v1_healthy_UMAP.pdf", dpi=200, bbox_inches = "tight")

In [ ]:
fig, ax = plt.subplots(figsize=(5,5))
sc.pl.umap(adata, 
           color=['celltype_v2'], 
           groups=atypical_ct,
           palette=all_colors, 
           na_color="#d3d3d3", 
           ax=ax, 
           show=False, 
           title="atypical colored",
           frameon=False)
ax.set_rasterization_zorder(0)
fig.savefig("figures/mastercolors_v1_atypical_UMAP.pdf", dpi=200, bbox_inches = "tight")

In [ ]:
adata

## Patient by patient

In [ ]:
newdf=pd.DataFrame({
    "X_coord_umap":adata.obsm["X_umap"][:,0],
    "Y_coord_umap":adata.obsm["X_umap"][:,1],
    "celltype":adata.obs["celltype_v2"],
    "outcome_C6D28":adata.obs["outcome_C6D28"],
    'outcome_C12D29':adata.obs["outcome_C12D29"],
    "timepoint_coarse":adata.obs["timepoint_coarse"],
    "patient_alias": adata.obs["patient_alias"],
})
newdf

In [ ]:
healthy_pat = ['C1','C2','C3','C4','C5']
newdf = newdf[~newdf['patient_alias'].isin(healthy_pat)]
newdf

In [ ]:
newdf['ctgrey'] = "#e2e2e2"

In [ ]:
newdf['patient_alias'].unique()

In [ ]:
df=newdf[newdf['patient_alias'] == 'P11']


In [ ]:
adata.obs['celltype_v2'].value_counts()

In [ ]:
new_celltype_colors2 = {
    "Atypical cluster a": "#808080",
    "Atypical cluster b": "#808080",
    "Atypical cluster c": "#808080",
    "Atypical cluster d": "#808080",
    "Atypical cluster e": "#808080",
    "Atypical cluster f": "#808080",
    "Atypical cluster g": "#808080",
    "Atypical cluster h": "#808080",
    "Atypical cluster i": "#808080",
    "Atypical cluster j": "#808080",
    "Atypical cluster k": "#808080",
    "Atypical cluster l": "#808080",
    "Atypical cluster m": "#808080",
    "Atypical cluster n": "#808080",
    "Atypical cluster O": "#808080",
    "HSC": "#c098b0",
    "MPP": "#583070",
    "MEP/MKP": "#ffb651",
    "MEP": "#d85a44",
    "Erythroblast": "#aa3f5d",
    "BMCP": "#bcd1d4",
    "GMP": "#79a3aa",
    "Monocyte progenitor": "#206671"
}

In [ ]:
#Get the list of patients
patientlist = newdf['patient_alias'].unique()

#for each patient plot the cells pre, mid, post and progression

for i, pat_a in enumerate(patientlist):

    df=newdf[newdf['patient_alias'] == pat_a]
    sb = len(df['timepoint_coarse'].unique())
    fig,axes = plt.subplots(1,sb, figsize=((6*sb),6), dpi=300)

    c=0
    for j, timepoint in enumerate(['pre','mid','post','progression']):
        if df['timepoint_coarse'].str.contains(timepoint).any():
            ax = axes[c]
            ax.scatter(newdf['X_coord_umap'], newdf['Y_coord_umap'], c=newdf['ctgrey'], s=1, )
            ax.scatter(df[df['timepoint_coarse'] == timepoint]['X_coord_umap'], 
                df[df['timepoint_coarse'] == timepoint]['Y_coord_umap'],
                c=[new_celltype_colors2[celltype] for celltype in df[df['timepoint_coarse'] == timepoint]['celltype']], 
                s=15,
                edgecolor='none')
            ax.set_rasterized(True)
            plottitle= pat_a + '-' + timepoint
            ax.set_title(plottitle)
            ax.grid(False)
            ax.legend().set_visible(False)
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_xlabel('')
            ax.set_ylabel('')
            ax.set_rasterization_zorder(0)
            ax.set(frame_on=False)
            c=c+1
        else:
            pass
        
    fig.tight_layout()
    fig.savefig("figures/"+str(pat_a)+"_timeplots_UMAP_v2.pdf", format="pdf", bbox_inches='tight')
    fig.savefig("figures/"+str(pat_a)+"_timeplots_UMAP_v2.png", format="png", bbox_inches='tight')

In [ ]:
#P17

df=newdf[newdf['patient_alias'] == "P17"]
sb = len(df['timepoint_coarse'].unique())
fig,axes = plt.subplots(1,sb, figsize=((6*sb),6), dpi=300)

c=0
for j, timepoint in enumerate(['pre','mid','post','progression']):
    if df['timepoint_coarse'].str.contains(timepoint).any():
        ax = axes[c]
        ax.scatter(newdf['X_coord_umap'], newdf['Y_coord_umap'], c=newdf['ctgrey'], s=1, )
        ax.scatter(df[df['timepoint_coarse'] == timepoint]['X_coord_umap'], 
            df[df['timepoint_coarse'] == timepoint]['Y_coord_umap'],
            c=[new_celltype_colors[celltype] for celltype in df[df['timepoint_coarse'] == timepoint]['celltype']], 
            s=15,
            edgecolor='none')
        plottitle= "P17" + '-' + timepoint
        ax.set_title(plottitle)
        ax.grid(False)
        ax.legend().set_visible(False)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_xlabel('')
        ax.set_ylabel('')
        ax.set_rasterization_zorder(0)
        ax.set(frame_on=False)
        c=c+1
    else:
        pass
    
fig.tight_layout()
fig.savefig("figures/P17_timeplots_allct_UMAP_v2.pdf", format="pdf", bbox_inches='tight')